# Claim Severity Modeling

## Objective

The goal of this notebook is to predict the financial size of an Auto insurance claim.

The target variable is `damage_amount`.

The modeling dataset comes from the `vw_auto_claim_severity_ml` SQL view.

The modeling process will compare different regression models and preprocessing strategies.

Because the target is right-skewed, both the original damage amount and a log-transformed target will be tested.

In [26]:
import pandas as pd
import numpy as np
import pyodbc

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
)
from sklearn.model_selection import RepeatedKFold

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    ElasticNet,
    HuberRegressor,
    GammaRegressor,
)
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
)
from xgboost import XGBRegressor

from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
conn = pyodbc.connect(
    "DSN=InsuranceAnalytics;",
    autocommit=True,
)

query = """
SELECT *
FROM vw_auto_claim_severity_ml
"""

df = pd.read_sql(
    query,
    conn,
)

df["occurrence_date"] = pd.to_datetime(
    df["occurrence_date"]
)

df["declaration_date"] = pd.to_datetime(
    df["declaration_date"]
)

print("Shape:", df.shape)
print("Claims:", df["claim_id"].nunique())
print("Average damage:", round(df["damage_amount"].mean(), 2))

Shape: (109, 24)
Claims: 109
Average damage: 4120.18


In [3]:
## Deterministic Feature Engineering

model_df = df.copy()

model_df["previous_claims_cat"] = (
    model_df["previous_claims"]
    .astype("Int64")
    .astype("string")
    .fillna("Unknown")
)

# Premium / vehicle value:
model_df["premium_value_ratio"] = (
    model_df["annual_premium"]
    / model_df["current_value"]
)

# Vehicle value / horsepower

model_df["value_per_hp"] = (
    model_df["current_value"]
    / model_df["power_hp"]
)

# Declaration lag transform
model_df["log_declaration_lag"] = np.log1p(
    model_df["declaration_lag_days"]
)

# Vehicle age group:
model_df["vehicle_age_group"] = pd.cut(
    model_df["vehicle_age_at_claim"],
    bins=[-1, 3, 7, np.inf],
    labels=[
        "0-3",
        "4-7",
        "8+",
    ],
).astype("object")

# Client age group
model_df["client_age_group"] = pd.cut(
    model_df["client_age"],
    bins=[
        17,
        29,
        39,
        49,
        59,
        np.inf,
    ],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60+",
    ],
).astype("object")

# Season:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"

    if month in [3, 4, 5]:
        return "Spring"

    if month in [6, 7, 8]:
        return "Summer"

    return "Autumn"


model_df["occurrence_season"] = (
    model_df["occurrence_date"]
    .dt.month
    .map(get_season)
)

# Target log:

model_df["log_damage_amount"] = np.log1p(
    model_df["damage_amount"]
)

# Control:
engineered_features = [
    "previous_claims_cat",
    "premium_value_ratio",
    "value_per_hp",
    "log_declaration_lag",
    "vehicle_age_group",
    "client_age_group",
    "occurrence_season",
]

model_df[
    engineered_features
].head()

,previous_claims_cat,premium_value_ratio,value_per_hp,log_declaration_lag,vehicle_age_group,client_age_group,occurrence_season
0,1,0.0465,100.8248,4.0254,0-3,30-39,Autumn
1,2,0.1188,45.0204,3.4012,8+,50-59,Autumn
2,0,0.1385,58.2783,3.5835,8+,18-29,Spring
3,0,0.0781,53.9766,3.4012,0-3,40-49,Winter
4,0,0.1067,67.0273,3.6636,4-7,30-39,Winter


In [4]:
core_numeric_features = [
    "annual_premium",
    "client_age",
    "declaration_lag_days",
    "power_hp",
    "current_value",
    "vehicle_age_at_claim",
]

core_categorical_features = [
    "claim_type",
    "risk_zone",
    "channel",
    "csp",
    "gender",
    "brand",
    "fuel_type",
    "vehicle_usage",
    "previous_claims_cat",
]

engineered_numeric_features = [
    "premium_value_ratio",
    "value_per_hp",
    "log_declaration_lag",
]

engineered_categorical_features = [
    "vehicle_age_group",
    "client_age_group",
    "occurrence_season",
]

In [5]:
CORE_FEATURES = (
    core_numeric_features
    + core_categorical_features
)

X = model_df[
    CORE_FEATURES
].copy()

y = model_df[
    "damage_amount"
].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget summary:")
print(y.describe())

X shape: (109, 15)
y shape: (109,)

Target summary:
count      109.0000
mean     4,120.1822
std      4,933.1970
min        190.0100
25%        848.0000
50%      2,102.5500
75%      4,806.7800
max     19,682.8800
Name: damage_amount, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

In [7]:
split_summary = pd.DataFrame(
    {
        "dataset": ["Train", "Test"],
        "claims": [
            len(X_train),
            len(X_test),
        ],
        "mean_damage": [
            y_train.mean(),
            y_test.mean(),
        ],
        "median_damage": [
            y_train.median(),
            y_test.median(),
        ],
        "max_damage": [
            y_train.max(),
            y_test.max(),
        ],
    }
)

split_summary

,dataset,claims,mean_damage,median_damage,max_damage
0,Train,87,"3,711.8616","2,102.5500","19,016.7500"
1,Test,22,"5,734.9045","2,335.5050","19,682.8800"


### Regression Train-Test Split

A simple random split creates a difference between the train and test damage distributions.

The test set has a much higher average damage amount because the target is strongly right-skewed.

To create a more representative test set, the damage amount is divided into quantile groups.

The train-test split is then stratified using these groups.

In [8]:
severity_bins = pd.qcut(
    y,
    q=5,
    labels=False,
    duplicates="drop",
)

severity_bins.value_counts().sort_index()

damage_amount
0    22
1    22
2    21
3    22
4    22
Name: count, dtype: int64

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=severity_bins,
    random_state=42,
)

In [10]:
split_summary = pd.DataFrame(
    {
        "dataset": ["Train", "Test"],
        "claims": [
            len(X_train),
            len(X_test),
        ],
        "mean_damage": [
            y_train.mean(),
            y_test.mean(),
        ],
        "median_damage": [
            y_train.median(),
            y_test.median(),
        ],
        "max_damage": [
            y_train.max(),
            y_test.max(),
        ],
        "damage_p90": [
            y_train.quantile(0.90),
            y_test.quantile(0.90),
        ],
    }
)

split_summary

,dataset,claims,mean_damage,median_damage,max_damage,damage_p90
0,Train,87,"4,225.4329","2,102.5500","19,682.8800","13,156.3800"
1,Test,22,"3,703.9636","2,091.4800","14,938.8800","8,134.0370"


### Final Train-Test Split

The quantile-stratified split gives a more balanced damage distribution than the simple random split.

The train and test median damage amounts are very similar.

Some difference remains in the upper tail because the test set contains only 22 claims.

The split will now be fixed, and the test set will not be used for preprocessing or model-selection decisions.

In [11]:
train_numeric = X_train[
    core_numeric_features
].copy()

severity_train_corr = (
    train_numeric
    .corr(method="spearman")
    .round(3)
)

severity_train_corr

,annual_premium,client_age,declaration_lag_days,power_hp,current_value,vehicle_age_at_claim
annual_premium,1.0000,-0.2130,-0.1100,0.1460,0.3360,-0.3540
client_age,-0.2130,1.0000,-0.0280,-0.0400,0.0440,-0.0860
declaration_lag_days,-0.1100,-0.0280,1.0000,-0.1570,0.0000,0.0170
power_hp,0.1460,-0.0400,-0.1570,1.0000,0.2980,-0.2560
current_value,0.3360,0.0440,0.0000,0.2980,1.0000,-0.8610
vehicle_age_at_claim,-0.3540,-0.0860,0.0170,-0.2560,-0.8610,1.0000


In [12]:
corr_abs = severity_train_corr.abs()

upper_triangle = corr_abs.where(
    np.triu(
        np.ones(corr_abs.shape),
        k=1,
    ).astype(bool)
)

high_corr_pairs = []

for column in upper_triangle.columns:
    for index in upper_triangle.index:
        value = upper_triangle.loc[index, column]

        if pd.notna(value) and value >= 0.80:
            high_corr_pairs.append(
                {
                    "feature_1": index,
                    "feature_2": column,
                    "abs_spearman_corr": value,
                }
            )

high_corr_pairs = pd.DataFrame(
    high_corr_pairs
)

high_corr_pairs

,feature_1,feature_2,abs_spearman_corr
0,current_value,vehicle_age_at_claim,0.8610


### Correlation Decision

`current_value` and `vehicle_age_at_claim` have a strong negative relationship.

However, the absolute correlation is below the 0.90 feature-removal threshold.

The variables also represent different business information.

For this reason, both features will remain in the initial feature set.

In [13]:
def outlier_summary_iqr(
    df,
    columns,
    multiplier=1.5,
):
    results = []

    for column in columns:
        values = df[column].dropna()

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - multiplier * iqr
        upper_bound = q3 + multiplier * iqr

        outlier_mask = (
            (values < lower_bound)
            | (values > upper_bound)
        )

        outlier_count = outlier_mask.sum()

        results.append(
            {
                "feature": column,
                "q1": q1,
                "q3": q3,
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "outlier_count": outlier_count,
                "outlier_pct": (
                    outlier_count
                    / len(values)
                    * 100
                ),
            }
        )

    return pd.DataFrame(results)

In [14]:
outlier_summary = outlier_summary_iqr(
    X_train,
    core_numeric_features,
)

outlier_summary

,feature,q1,q3,lower_bound,upper_bound,outlier_count,outlier_pct
0,annual_premium,635.0550,"1,051.3600",10.5975,"1,675.8175",0,0.0000
1,client_age,37.2500,52.7500,14.0000,76.0000,0,0.0000
2,declaration_lag_days,8.0000,34.7500,-32.1250,74.8750,0,0.0000
3,power_hp,127.6800,178.8900,50.8650,255.7050,0,0.0000
4,current_value,"5,533.2750","12,739.7350","-5,276.4150","23,549.4250",4,4.5977
5,vehicle_age_at_claim,2.0000,7.0000,-5.5000,14.5000,0,0.0000


### Outlier Findings

Most numeric features do not contain IQR-based outliers in the training data.

`current_value` contains about 4.6% outliers.

These observations will not be removed because high-value vehicles can represent valid insurance records.

For preprocessing, `current_value` will use RobustScaler. The other numeric features will use StandardScaler.

In [16]:
standard_columns = [
    "annual_premium",
    "client_age",
    "declaration_lag_days",
    "power_hp",
    "vehicle_age_at_claim",
]

robust_columns = [
    "current_value",
]

onehot_columns = [
    "claim_type",
    "risk_zone",
    "channel",
    "csp",
    "gender",
    "brand",
    "fuel_type",
    "vehicle_usage",
    "previous_claims_cat",
]

In [18]:
standard_transformer = Pipeline(
    steps=[
        (
            "median_imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "standard_scaler",
            StandardScaler(),
        ),
    ]
)

robust_transformer = Pipeline(
    steps=[
        (
            "median_imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "robust_scaler",
            RobustScaler(),
        ),
    ]
)

In [19]:
categorical_transformer = Pipeline(
    steps=[
        (
            "unknown_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown",
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

In [20]:
preprocess = ColumnTransformer(
    transformers=[
        (
            "standard_num",
            standard_transformer,
            standard_columns,
        ),
        (
            "robust_num",
            robust_transformer,
            robust_columns,
        ),
        (
            "categorical",
            categorical_transformer,
            onehot_columns,
        ),
    ],
    remainder="drop",
)

In [21]:
preprocessing_columns = (
    standard_columns
    + robust_columns
    + onehot_columns
)

print("X_train features:", len(X_train.columns))
print("Preprocessing features:", len(preprocessing_columns))

print(
    "Missing from preprocessing:",
    set(X_train.columns) - set(preprocessing_columns),
)

print(
    "Unexpected columns:",
    set(preprocessing_columns) - set(X_train.columns),
)

X_train features: 15
Preprocessing features: 15
Missing from preprocessing: set()
Unexpected columns: set()


## Baseline Regression Models

Several regression models will be compared using the same preprocessing pipeline.

Because the severity dataset is small, repeated cross-validation will be used for a more stable evaluation.

The main metric is MAE because it is easy to interpret in the original damage amount scale.

RMSE will also be used because it gives more importance to large prediction errors.

A simple median prediction will be included as a baseline.

In [23]:
repeated_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

In [25]:
models = {
    "Dummy_Median": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                DummyRegressor(
                    strategy="median",
                ),
            ),
        ]
    ),

    "LinearRegression": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                LinearRegression(),
            ),
        ]
    ),

    "Ridge": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                Ridge(
                    alpha=1.0,
                ),
            ),
        ]
    ),

    "ElasticNet": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                ElasticNet(
                    alpha=0.1,
                    l1_ratio=0.5,
                    max_iter=5000,
                    random_state=42,
                ),
            ),
        ]
    ),

    "Huber": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                HuberRegressor(
                    max_iter=5000,
                ),
            ),
        ]
    ),

    "Gamma": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                GammaRegressor(
                    alpha=1.0,
                    max_iter=5000,
                ),
            ),
        ]
    ),

    "RandomForest": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    min_samples_leaf=3,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),

    "ExtraTrees": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=300,
                    min_samples_leaf=3,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),

    "GradientBoosting": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                GradientBoostingRegressor(
                    n_estimators=200,
                    learning_rate=0.03,
                    max_depth=2,
                    random_state=42,
                ),
            ),
        ]
    ),

    "XGBoost": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                XGBRegressor(
                    n_estimators=200,
                    max_depth=2,
                    learning_rate=0.03,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="reg:squarederror",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

In [27]:
baseline_results = []

for model_name, model_pipeline in models.items():

    for fold, (train_idx, val_idx) in enumerate(
        repeated_cv.split(X_train),
        start=1,
    ):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        fold_model = clone(model_pipeline)

        fold_model.fit(
            X_tr,
            y_tr,
        )

        val_pred = fold_model.predict(
            X_val
        )

        baseline_results.append(
            {
                "model": model_name,
                "fold": fold,

                "mae": mean_absolute_error(
                    y_val,
                    val_pred,
                ),

                "rmse": np.sqrt(
                    mean_squared_error(
                        y_val,
                        val_pred,
                    )
                ),

                "r2": r2_score(
                    y_val,
                    val_pred,
                ),
            }
        )

baseline_results = pd.DataFrame(
    baseline_results
)

In [28]:
baseline_summary = (
    baseline_results
    .groupby("model")
    .agg(
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        mae_median=("mae", "median"),

        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),

        r2_mean=("r2", "mean"),
    )
    .sort_values(
        "mae_mean",
        ascending=True,
    )
)

baseline_summary

,mae_mean,mae_std,mae_median,rmse_mean,rmse_std,r2_mean
model,,,,,,
RandomForest,"1,117.3973",322.3331,"1,156.6444","1,799.1273",591.2560,0.8324
GradientBoosting,"1,297.4509",285.3498,"1,283.6999","1,982.5052",485.5407,0.7971
XGBoost,"1,302.4416",308.7713,"1,229.5974","1,972.5406",566.5182,0.8090
ExtraTrees,"1,399.9646",366.0915,"1,387.7063","2,155.1126",623.3143,0.7680
Huber,"1,456.5286",345.7452,"1,393.6394","2,038.5842",587.7286,0.7945
Ridge,"1,725.3868",449.5174,"1,693.5156","2,347.4740",685.3699,0.7244
ElasticNet,"1,937.3381",524.0585,"1,848.7682","2,581.9010",796.1421,0.6879
LinearRegression,"1,970.1311",417.7749,"1,894.5889","2,612.0956",595.0428,0.6361
Gamma,"3,269.6544",815.4674,"3,257.4483","4,629.3891","1,288.4134",0.0817


### Baseline Model Results

Tree-based models perform much better than the simple median baseline.

Random Forest has the lowest MAE and the highest R² among the tested models.

Gradient Boosting and XGBoost also perform well.

Linear models are weaker, which suggests that claim severity contains non-linear relationships.

The strong performance may be partly explained by `claim_type`, which showed a very strong relationship with damage amount during EDA.

In [29]:
claim_type_baseline_results = []

for fold, (train_idx, val_idx) in enumerate(
    repeated_cv.split(X_train),
    start=1,
):
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    train_temp = pd.DataFrame(
        {
            "claim_type": X_tr["claim_type"],
            "damage_amount": y_tr,
        }
    )

    claim_type_medians = (
        train_temp
        .groupby("claim_type")["damage_amount"]
        .median()
    )

    global_median = y_tr.median()

    val_pred = (
        X_val["claim_type"]
        .map(claim_type_medians)
        .fillna(global_median)
        .to_numpy()
    )

    claim_type_baseline_results.append(
        {
            "fold": fold,
            "mae": mean_absolute_error(
                y_val,
                val_pred,
            ),
            "rmse": np.sqrt(
                mean_squared_error(
                    y_val,
                    val_pred,
                )
            ),
            "r2": r2_score(
                y_val,
                val_pred,
            ),
        }
    )

claim_type_baseline_results = pd.DataFrame(
    claim_type_baseline_results
)

In [30]:
claim_type_baseline_summary = pd.Series(
    {
        "mae_mean": (
            claim_type_baseline_results["mae"].mean()
        ),
        "mae_std": (
            claim_type_baseline_results["mae"].std()
        ),
        "mae_median": (
            claim_type_baseline_results["mae"].median()
        ),
        "rmse_mean": (
            claim_type_baseline_results["rmse"].mean()
        ),
        "r2_mean": (
            claim_type_baseline_results["r2"].mean()
        ),
    }
)

claim_type_baseline_summary

mae_mean     1,062.5229
mae_std        306.8859
mae_median   1,055.8056
rmse_mean    1,729.7369
r2_mean          0.8455
dtype: float64

### Claim Type Baseline

Claim type alone provides a very strong severity baseline.

Using the median damage amount for each claim type gives a lower MAE than the tested machine-learning models.

This shows that a large part of claim severity is explained by claim type.

The other customer and vehicle features may provide only limited additional information.

For this reason, claim type will be used as an important business benchmark for the severity models.

In [31]:
onehot_columns_no_claim_type = [
    col
    for col in onehot_columns
    if col != "claim_type"
]

In [32]:
preprocess_no_claim_type = ColumnTransformer(
    transformers=[
        (
            "standard_num",
            standard_transformer,
            standard_columns,
        ),
        (
            "robust_num",
            robust_transformer,
            robust_columns,
        ),
        (
            "categorical",
            categorical_transformer,
            onehot_columns_no_claim_type,
        ),
    ],
    remainder="drop",
)

In [33]:
rf_no_claim_type = Pipeline(
    steps=[
        (
            "preprocess",
            preprocess_no_claim_type,
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [34]:
ablation_results = []

for fold, (train_idx, val_idx) in enumerate(
    repeated_cv.split(X_train),
    start=1,
):
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    fold_model = clone(rf_no_claim_type)

    fold_model.fit(
        X_tr,
        y_tr,
    )

    val_pred = fold_model.predict(
        X_val
    )

    ablation_results.append(
        {
            "fold": fold,
            "mae": mean_absolute_error(
                y_val,
                val_pred,
            ),
            "rmse": np.sqrt(
                mean_squared_error(
                    y_val,
                    val_pred,
                )
            ),
            "r2": r2_score(
                y_val,
                val_pred,
            ),
        }
    )

ablation_results = pd.DataFrame(
    ablation_results
)

In [35]:
ablation_summary = pd.Series(
    {
        "mae_mean": ablation_results["mae"].mean(),
        "mae_std": ablation_results["mae"].std(),
        "rmse_mean": ablation_results["rmse"].mean(),
        "r2_mean": ablation_results["r2"].mean(),
    }
)

ablation_summary

mae_mean    4,446.7847
mae_std       650.3736
rmse_mean   5,562.0893
r2_mean        -0.4345
dtype: float64

### Claim Type Ablation Test

Removing `claim_type` causes a large decrease in model performance.

The Random Forest model without claim type has a negative R² and a much higher MAE.

This confirms that `claim_type` is the main predictive feature for claim severity.

Customer and vehicle features alone do not provide enough information to predict damage amount reliably.

In [36]:
residual_results = []

for fold, (train_idx, val_idx) in enumerate(
    repeated_cv.split(X_train),
    start=1,
):
    X_tr = X_train.iloc[train_idx].copy()
    X_val = X_train.iloc[val_idx].copy()

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Claim-type baseline from training data
    train_temp = pd.DataFrame(
        {
            "claim_type": X_tr["claim_type"],
            "damage_amount": y_tr,
        }
    )

    claim_type_medians = (
        train_temp
        .groupby("claim_type")["damage_amount"]
        .median()
    )

    global_median = y_tr.median()

    train_baseline = (
        X_tr["claim_type"]
        .map(claim_type_medians)
        .fillna(global_median)
        .to_numpy()
    )

    val_baseline = (
        X_val["claim_type"]
        .map(claim_type_medians)
        .fillna(global_median)
        .to_numpy()
    )

    # Residual target
    y_residual = (
        y_tr.to_numpy()
        - train_baseline
    )

    # Claim type removed because it is already
    # represented by the baseline
    residual_model = clone(
        rf_no_claim_type
    )

    residual_model.fit(
        X_tr,
        y_residual,
    )

    residual_pred = residual_model.predict(
        X_val
    )

    final_pred = (
        val_baseline
        + residual_pred
    )

    residual_results.append(
        {
            "fold": fold,
            "mae": mean_absolute_error(
                y_val,
                final_pred,
            ),
            "rmse": np.sqrt(
                mean_squared_error(
                    y_val,
                    final_pred,
                )
            ),
            "r2": r2_score(
                y_val,
                final_pred,
            ),
        }
    )

residual_results = pd.DataFrame(
    residual_results
)

In [37]:
residual_summary = pd.Series(
    {
        "mae_mean": residual_results["mae"].mean(),
        "mae_std": residual_results["mae"].std(),
        "mae_median": residual_results["mae"].median(),
        "rmse_mean": residual_results["rmse"].mean(),
        "r2_mean": residual_results["r2"].mean(),
    }
)

residual_summary

mae_mean     1,212.8963
mae_std        281.8326
mae_median   1,167.0722
rmse_mean    1,791.4436
r2_mean          0.8389
dtype: float64

### Residual Modeling Result

The residual Random Forest does not improve the claim-type median baseline.

The claim-type baseline still has a lower MAE and RMSE.

This suggests that customer and vehicle features do not provide stable additional information after claim type is known.

Claim type remains the strongest severity predictor and an important benchmark for the final model.

In [38]:
engineered_numeric_features = [
    "premium_value_ratio",
    "value_per_hp",
    "log_declaration_lag",
]

engineered_categorical_features = [
    "vehicle_age_group",
    "client_age_group",
    "occurrence_season",
]

EXTENDED_FEATURES = (
    core_numeric_features
    + core_categorical_features
    + engineered_numeric_features
    + engineered_categorical_features
)

X_extended = model_df[EXTENDED_FEATURES].copy()

X_train_extended = X_extended.loc[X_train.index]
X_test_extended = X_extended.loc[X_test.index]

print("Core train shape:", X_train.shape)
print("Extended train shape:", X_train_extended.shape)

Core train shape: (87, 15)
Extended train shape: (87, 21)


In [39]:
engineered_outlier_summary = outlier_summary_iqr(
    X_train_extended,
    engineered_numeric_features,
)

engineered_outlier_summary

,feature,q1,q3,lower_bound,upper_bound,outlier_count,outlier_pct
0,premium_value_ratio,0.0609,0.1250,-0.0351,0.2211,4,4.5977
1,value_per_hp,38.6994,91.8444,-41.0181,171.5619,1,1.3889
2,log_declaration_lag,2.1972,3.5765,0.1283,5.6454,3,3.4884


In [40]:
extended_numeric_features = (
    core_numeric_features
    + engineered_numeric_features
)

extended_corr = (
    X_train_extended[extended_numeric_features]
    .corr(method="spearman")
    .round(3)
)

extended_corr

,annual_premium,client_age,declaration_lag_days,power_hp,current_value,vehicle_age_at_claim,premium_value_ratio,value_per_hp,log_declaration_lag
annual_premium,1.0000,-0.2130,-0.1100,0.1460,0.3360,-0.3540,0.1800,0.3640,-0.1100
client_age,-0.2130,1.0000,-0.0280,-0.0400,0.0440,-0.0860,-0.1280,0.0210,-0.0280
declaration_lag_days,-0.1100,-0.0280,1.0000,-0.1570,0.0000,0.0170,-0.0620,0.0120,1.0000
power_hp,0.1460,-0.0400,-0.1570,1.0000,0.2980,-0.2560,-0.2640,-0.0850,-0.1570
current_value,0.3360,0.0440,0.0000,0.2980,1.0000,-0.8610,-0.8410,0.9080,0.0000
vehicle_age_at_claim,-0.3540,-0.0860,0.0170,-0.2560,-0.8610,1.0000,0.6850,-0.8170,0.0170
premium_value_ratio,0.1800,-0.1280,-0.0620,-0.2640,-0.8410,0.6850,1.0000,-0.7330,-0.0620
value_per_hp,0.3640,0.0210,0.0120,-0.0850,0.9080,-0.8170,-0.7330,1.0000,0.0120
log_declaration_lag,-0.1100,-0.0280,1.0000,-0.1570,0.0000,0.0170,-0.0620,0.0120,1.0000


### Engineered Feature Review

Some engineered features are strongly related to the original variables.

`value_per_hp` has a correlation above 0.90 with `current_value`, so it will not be used.

`log_declaration_lag` contains the same ranking information as `declaration_lag_days`. Therefore, the log version will replace the original variable in the extended feature set.

`premium_value_ratio` will be kept because it represents the premium relative to the insured vehicle value.

The categorical engineered features will also be tested.

In [41]:
extended_standard_columns = [
    "annual_premium",
    "client_age",
    "power_hp",
    "vehicle_age_at_claim",
]

In [44]:
extended_robust_columns = [
    "current_value",
    "premium_value_ratio",
    "log_declaration_lag",
]

In [43]:
extended_onehot_columns = [
    "claim_type",
    "risk_zone",
    "channel",
    "csp",
    "gender",
    "brand",
    "fuel_type",
    "vehicle_usage",
    "previous_claims_cat",
    "vehicle_age_group",
    "client_age_group",
    "occurrence_season",
]

In [45]:
extended_preprocess = ColumnTransformer(
    transformers=[
        (
            "standard_num",
            standard_transformer,
            extended_standard_columns,
        ),
        (
            "robust_num",
            robust_transformer,
            extended_robust_columns,
        ),
        (
            "categorical",
            categorical_transformer,
            extended_onehot_columns,
        ),
    ],
    remainder="drop",
)

In [46]:
EXTENDED_FEATURES = (
    extended_standard_columns
    + extended_robust_columns
    + extended_onehot_columns
)

X_extended = model_df[
    EXTENDED_FEATURES
].copy()

X_train_extended = X_extended.loc[
    X_train.index
]

X_test_extended = X_extended.loc[
    X_test.index
]

In [47]:
extended_preprocessing_columns = (
    extended_standard_columns
    + extended_robust_columns
    + extended_onehot_columns
)

print(
    "Extended X_train features:",
    len(X_train_extended.columns),
)

print(
    "Preprocessing features:",
    len(extended_preprocessing_columns),
)

print(
    "Missing from preprocessing:",
    set(X_train_extended.columns)
    - set(extended_preprocessing_columns),
)

print(
    "Unexpected columns:",
    set(extended_preprocessing_columns)
    - set(X_train_extended.columns),
)

Extended X_train features: 19
Preprocessing features: 19
Missing from preprocessing: set()
Unexpected columns: set()


## Core, Extended, and Log-Target Comparison

The best tree-based model families are tested with different feature and target strategies.

Three approaches are compared:

- Core features with the original damage target.
- Extended features with the original damage target.
- Extended features with a log-transformed damage target.

All predictions are evaluated on the original damage amount scale.

The claim-type median baseline remains an important business benchmark.

In [48]:
selected_regressors = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        random_state=42,
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    ),
}

In [49]:
comparison_results = []

for fold, (train_idx, val_idx) in enumerate(
    repeated_cv.split(X_train),
    start=1,
):

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Core feature data
    X_tr_core = X_train.iloc[train_idx]
    X_val_core = X_train.iloc[val_idx]

    # Extended feature data
    X_tr_extended = X_train_extended.iloc[train_idx]
    X_val_extended = X_train_extended.iloc[val_idx]

    for model_name, regressor in selected_regressors.items():

        # ---------------------------------
        # 1. Core + Raw Target
        # ---------------------------------
        core_model = Pipeline(
            steps=[
                ("preprocess", preprocess),
                ("model", clone(regressor)),
            ]
        )

        core_model.fit(
            X_tr_core,
            y_tr,
        )

        core_pred = core_model.predict(
            X_val_core
        )

        comparison_results.append(
            {
                "model": model_name,
                "strategy": "Core_Raw",
                "fold": fold,
                "mae": mean_absolute_error(
                    y_val,
                    core_pred,
                ),
                "rmse": np.sqrt(
                    mean_squared_error(
                        y_val,
                        core_pred,
                    )
                ),
                "r2": r2_score(
                    y_val,
                    core_pred,
                ),
            }
        )

        # ---------------------------------
        # 2. Extended + Raw Target
        # ---------------------------------
        extended_model = Pipeline(
            steps=[
                (
                    "preprocess",
                    extended_preprocess,
                ),
                ("model", clone(regressor)),
            ]
        )

        extended_model.fit(
            X_tr_extended,
            y_tr,
        )

        extended_pred = extended_model.predict(
            X_val_extended
        )

        comparison_results.append(
            {
                "model": model_name,
                "strategy": "Extended_Raw",
                "fold": fold,
                "mae": mean_absolute_error(
                    y_val,
                    extended_pred,
                ),
                "rmse": np.sqrt(
                    mean_squared_error(
                        y_val,
                        extended_pred,
                    )
                ),
                "r2": r2_score(
                    y_val,
                    extended_pred,
                ),
            }
        )

        # ---------------------------------
        # 3. Extended + Log Target
        # ---------------------------------
        log_model = Pipeline(
            steps=[
                (
                    "preprocess",
                    extended_preprocess,
                ),
                ("model", clone(regressor)),
            ]
        )

        y_tr_log = np.log1p(y_tr)

        log_model.fit(
            X_tr_extended,
            y_tr_log,
        )

        log_pred = log_model.predict(
            X_val_extended
        )

        # Back to original money scale
        log_pred_original = np.expm1(
            log_pred
        )

        comparison_results.append(
            {
                "model": model_name,
                "strategy": "Extended_Log",
                "fold": fold,
                "mae": mean_absolute_error(
                    y_val,
                    log_pred_original,
                ),
                "rmse": np.sqrt(
                    mean_squared_error(
                        y_val,
                        log_pred_original,
                    )
                ),
                "r2": r2_score(
                    y_val,
                    log_pred_original,
                ),
            }
        )

comparison_results = pd.DataFrame(
    comparison_results
)

In [50]:
comparison_summary = (
    comparison_results
    .groupby(
        [
            "model",
            "strategy",
        ]
    )
    .agg(
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        mae_median=("mae", "median"),

        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),

        r2_mean=("r2", "mean"),
    )
    .sort_values(
        "mae_mean",
        ascending=True,
    )
)

comparison_summary

mae_mean  mae_std  mae_median  rmse_mean  \
model            strategy                                                  
RandomForest     Extended_Log 1,102.0263 339.0641  1,146.3190 1,834.2714   
                 Core_Raw     1,117.3973 322.3331  1,156.6444 1,799.1273   
                 Extended_Raw 1,123.2314 326.8327  1,147.9434 1,808.2412   
XGBoost          Extended_Log 1,293.0470 449.6687  1,197.3232 2,170.1559   
GradientBoosting Extended_Log 1,295.9587 469.2246  1,131.3795 2,195.4130   
                 Extended_Raw 1,296.0394 296.9583  1,288.5225 1,948.3094   
                 Core_Raw     1,297.4509 285.3498  1,283.6999 1,982.5052   
XGBoost          Extended_Raw 1,300.2693 329.8059  1,301.1581 1,959.6259   
                 Core_Raw     1,302.4416 308.7713  1,229.5974 1,972.5406   

                               rmse_std  r2_mean  
model            strategy                         
RandomForest     Extended_Log  614.0511   0.8256  
                 Core_Raw      591.2560   0.8324  
                 Extended_Raw  591.3737   0.8303  
XGBoost          Extended_Log  829.4383   0.7796  
GradientBoosting Extended_Log  855.8587   0.7736  
                 Extended_Raw  535.3294   0.8031  
                 Core_Raw      485.5407   0.7971  
XGBoost          Extended_Raw  579.3575   0.8092  
                 Core_Raw      566.5182   0.8090

### Feature and Target Comparison

The engineered features do not provide a clear improvement over the core feature set.

The log-transformed target slightly improves the Random Forest MAE, but the improvement is small.

The claim-type median baseline still has the lowest cross-validation MAE.

This confirms that claim type explains most of the predictable variation in claim severity.

More model tuning is unlikely to provide a reliable improvement with the current small dataset.

In [51]:
train_claim_type_df = pd.DataFrame(
    {
        "claim_type": X_train["claim_type"],
        "damage_amount": y_train,
    }
)

claim_type_medians = (
    train_claim_type_df
    .groupby("claim_type")["damage_amount"]
    .median()
)

global_train_median = y_train.median()

claim_type_test_pred = (
    X_test["claim_type"]
    .map(claim_type_medians)
    .fillna(global_train_median)
    .to_numpy()
)

In [53]:
claim_type_test_mae = mean_absolute_error(
    y_test,
    claim_type_test_pred,
)

claim_type_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        claim_type_test_pred,
    )
)

claim_type_test_r2 = r2_score(
    y_test,
    claim_type_test_pred,
)

print("Claim Type Test MAE:", round(claim_type_test_mae, 2))
print("Claim Type Test RMSE:", round(claim_type_test_rmse, 2))
print("Claim Type Test R2:", round(claim_type_test_r2, 4))

Claim Type Test MAE: 1046.47
Claim Type Test RMSE: 1495.96
Claim Type Test R2: 0.8687


### Claim Type Baseline Test Result

The claim-type median baseline performs well on the untouched test set.

The test MAE is about 1,046 and the test R² is about 0.87.

These results are consistent with the cross-validation results.

This confirms that claim type is a strong and stable predictor of claim severity.

In [54]:
final_rf_log = Pipeline(
    steps=[
        (
            "preprocess",
            extended_preprocess,
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

y_train_log = np.log1p(y_train)

final_rf_log.fit(
    X_train_extended,
    y_train_log,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['annual_premium','client_age','power_hp',...,'vehicle_age_group', 'client_age_group','occurrence_season']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standard_num', ...), ('robust_num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will b

In [55]:
rf_test_log_pred = final_rf_log.predict(
    X_test_extended
)

rf_test_pred = np.expm1(
    rf_test_log_pred
)

In [56]:
rf_test_mae = mean_absolute_error(
    y_test,
    rf_test_pred,
)

rf_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_test_pred,
    )
)

rf_test_r2 = r2_score(
    y_test,
    rf_test_pred,
)

print(
    "RF Extended Log Test MAE:",
    round(rf_test_mae, 2),
)

print(
    "RF Extended Log Test RMSE:",
    round(rf_test_rmse, 2),
)

print(
    "RF Extended Log Test R2:",
    round(rf_test_r2, 4),
)

RF Extended Log Test MAE: 973.6
RF Extended Log Test RMSE: 1473.34
RF Extended Log Test R2: 0.8727


## Final Severity Model Evaluation

The Random Forest model with extended features and a log-transformed target performs well on the untouched test set.

Its test MAE is about 974 and its test R² is about 0.87.

The Random Forest performs slightly better than the claim-type median baseline on the test set.

However, the claim-type baseline performed slightly better during repeated cross-validation.

For this reason, the Random Forest is selected as the final machine-learning severity model, while the claim-type median remains an important and strong business benchmark.

The results also show that `claim_type` explains a large part of claim severity.

## Modeling Conclusion

Claim severity can be predicted much better than claim occurrence in the current dataset.

`claim_type` is the strongest predictor of damage amount.

Tree-based models perform better than linear models. Random Forest gives the best overall machine-learning results.

Feature engineering and the log-transformed target provide a small improvement in the final test performance.

The selected machine-learning model is:

- Random Forest Regressor
- Extended feature set
- `log1p(damage_amount)` target

Predictions are transformed back to the original damage scale for evaluation.

Because the dataset contains only 109 claims, the results should still be interpreted carefully.